In [ ]:
import sqlite3
from datetime import datetime

# Connect to SQLite database (or create it if not exists)
conn = sqlite3.connect("expenses.db")
cursor = conn.cursor()

# Create expenses table if it doesn't already exist
cursor.execute("""
CREATE TABLE IF NOT EXISTS expenses (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    date TEXT,
    category TEXT,
    amount REAL,
    description TEXT
)
""")
conn.commit()

# Function to add a new expense
def add_expense(date, category, amount, description):
    cursor.execute(
        "INSERT INTO expenses (date, category, amount, description) VALUES (?, ?, ?, ?)",
        (date, category, amount, description)
    )
    conn.commit()

# Function to view all expenses or within a date range
def view_expenses(start_date=None, end_date=None):
    if start_date and end_date:
        # Fetch only records within the date range
        cursor.execute(
            "SELECT * FROM expenses WHERE date BETWEEN ? AND ? ORDER BY date",
            (start_date, end_date)
        )
    else:
        # Fetch all records
        cursor.execute("SELECT * FROM expenses ORDER BY date")

    rows = cursor.fetchall()
    if not rows:
        print("No expense records found.")
    else:
        print("\n--- Expense Records ---")
        for row in rows:
            print(f"Date: {row[1]}, Category: {row[2]}, Amount: ₦{row[3]:,.2f}, Description: {row[4]}")
        print("------------------------")

# Function to summarize total amount spent per category
def summarize_by_category():
    cursor.execute("""
    SELECT category, SUM(amount) FROM expenses GROUP BY category
    """)
    summary = cursor.fetchall()
    if summary:
        print("\n--- Spending Summary by Category ---")
        for cat, total in summary:
            print(f"{cat}: ₦{total:,.2f}")
        print("-------------------------------------")
    else:
        print("No data to summarize.")

# Get today's date as default
date = datetime.today().strftime("%Y-%m-%d")

# Ask user for expense details
category = input("Enter expense category: ")
amount = float(input("Enter amount spent: "))
description = input("Enter description: ")

# Add expense to database
add_expense(date, category, amount, description)
print("✅ Expense added successfully!")

# Ask if user wants to filter by date
filter_choice = input("\nDo you want to filter by date range? (yes/no): ").strip().lower()
if filter_choice == "yes":
    start = input("Enter start date (YYYY-MM-DD): ")
    end = input("Enter end date (YYYY-MM-DD): ")
    view_expenses(start, end)
else:
    view_expenses()

# Show summary report
summarize_by_category()

# Close the database connection
conn.close()